# Retail Sales Performance & Profitability Analysis
**IBM SkillsBuild Data Analytics with AI Academic Internship Program**

**Objective:** Analyze retail orders to understand sales, profit, customer segments, product categories, regional performance, and discount patterns. The notebook includes data-quality checks and generates evidence-based, rule-based observations that can be reviewed or further interpreted with an AI assistant.

**Dataset:** [Sample Superstore CSV on GitHub](https://raw.githubusercontent.com/plotly/datasets/master/superstore.csv)

**Workflow:** Load → inspect → clean → validate → analyze → visualize → summarize → recommend.

> Run all cells from top to bottom. The notebook downloads the public dataset when an internet connection is available. If the source URL changes, download the CSV manually and update `DATA_URL` below.


## 1. Import libraries and configure the dataset

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_URL = "https://raw.githubusercontent.com/plotly/datasets/master/superstore.csv"
LOCAL_CSV = Path("superstore.csv")


## 2. Load data

In [ ]:
def load_data():
    """Load the dataset from a local CSV first, then try the public URL."""
    if LOCAL_CSV.exists():
        return pd.read_csv(LOCAL_CSV)
    try:
        return pd.read_csv(DATA_URL)
    except Exception as exc:
        raise RuntimeError(
            "Could not load the dataset. Check your internet connection or download "
            "the CSV from the README dataset link and save it as 'superstore.csv' "
            "in the notebook folder."
        ) from exc

df_raw = load_data()
df = df_raw.copy()

print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
display(df.head())


## 3. Initial data inspection

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nData types and non-null counts:")
df.info()

print("\nMissing values by column:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))

print("\nExact duplicate rows:", int(df.duplicated().sum()))
display(df.describe(include="all").T)


## 4. Data cleaning and validation
Normalize column labels, parse dates, remove exact duplicate records, and validate important numeric fields. Missing values are reported rather than silently imputed.

In [ ]:
# Normalize column labels to make later references consistent.
df.columns = [c.strip() for c in df.columns]

# Parse dates if the expected columns exist.
for col in ["Order Date", "Ship Date"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

# Convert key numeric columns safely.
for col in ["Sales", "Profit", "Quantity", "Discount"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

rows_before = len(df)
df = df.drop_duplicates().copy()
duplicates_removed = rows_before - len(df)

required_cols = ["Order Date", "Sales", "Profit", "Quantity", "Category", "Region"]
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f"Dataset is missing required columns: {missing_required}")

# Keep records with valid dates and core financial/quantity measures for time and KPI analysis.
valid_mask = (
    df["Order Date"].notna()
    & df["Sales"].notna()
    & df["Profit"].notna()
    & df["Quantity"].notna()
)
invalid_core_rows = int((~valid_mask).sum())
analysis_df = df.loc[valid_mask].copy()

print(f"Exact duplicates removed: {duplicates_removed:,}")
print(f"Rows excluded due to invalid date/sales/profit/quantity: {invalid_core_rows:,}")
print(f"Rows retained for analysis: {len(analysis_df):,}")
print("Validation: Sales, Profit, and Quantity are numeric:",
      all(pd.api.types.is_numeric_dtype(analysis_df[c]) for c in ["Sales", "Profit", "Quantity"]))
display(analysis_df.isna().sum().sort_values(ascending=False).to_frame("missing_after_cleaning"))


## 5. Key performance indicators (KPIs)

In [ ]:
total_sales = analysis_df["Sales"].sum()
total_profit = analysis_df["Profit"].sum()
total_quantity = analysis_df["Quantity"].sum()
profit_margin = (total_profit / total_sales * 100) if total_sales else np.nan

if "Order ID" in analysis_df.columns:
    total_orders = analysis_df["Order ID"].nunique()
else:
    total_orders = len(analysis_df)

if "Customer ID" in analysis_df.columns:
    total_customers = analysis_df["Customer ID"].nunique()
else:
    total_customers = np.nan

kpis = pd.DataFrame({
    "Metric": ["Total sales", "Total profit", "Profit margin (%)", "Units sold", "Orders", "Unique customers"],
    "Value": [total_sales, total_profit, profit_margin, total_quantity, total_orders, total_customers]
})
display(kpis)


## 6. Monthly sales and profit trends

In [ ]:
monthly = (
    analysis_df.assign(Month=analysis_df["Order Date"].dt.to_period("M").dt.to_timestamp())
    .groupby("Month", as_index=False)
    .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"))
    .sort_values("Month")
)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly["Month"], monthly["Sales"], marker="o", label="Sales")
ax.set_title("Monthly Sales Trend")
ax.set_xlabel("Month")
ax.set_ylabel("Sales")
ax.grid(True, alpha=0.3)
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly["Month"], monthly["Profit"], marker="o", label="Profit")
ax.axhline(0, linewidth=1)
ax.set_title("Monthly Profit Trend")
ax.set_xlabel("Month")
ax.set_ylabel("Profit")
ax.grid(True, alpha=0.3)
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

display(monthly.tail(12))


## 7. Category and sub-category performance

In [ ]:
category_perf = (
    analysis_df.groupby("Category", as_index=False)
    .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Units=("Quantity", "sum"))
    .sort_values("Sales", ascending=False)
)
category_perf["Profit Margin (%)"] = np.where(
    category_perf["Sales"] != 0,
    category_perf["Profit"] / category_perf["Sales"] * 100,
    np.nan
)
display(category_perf)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(category_perf["Category"].astype(str), category_perf["Sales"])
ax.set_title("Sales by Category")
ax.set_xlabel("Category")
ax.set_ylabel("Sales")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

subcategory_perf = (
    analysis_df.groupby("Sub-Category", as_index=False)
    .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"))
    .sort_values("Profit")
)
display(subcategory_perf.head(10))


## 8. Regional and customer-segment analysis

In [ ]:
region_perf = (
    analysis_df.groupby("Region", as_index=False)
    .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Units=("Quantity", "sum"))
    .sort_values("Sales", ascending=False)
)
region_perf["Profit Margin (%)"] = np.where(
    region_perf["Sales"] != 0,
    region_perf["Profit"] / region_perf["Sales"] * 100,
    np.nan
)
display(region_perf)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(region_perf["Region"].astype(str), region_perf["Profit"])
ax.axhline(0, linewidth=1)
ax.set_title("Profit by Region")
ax.set_xlabel("Region")
ax.set_ylabel("Profit")
plt.tight_layout()
plt.show()

if "Segment" in analysis_df.columns:
    segment_perf = (
        analysis_df.groupby("Segment", as_index=False)
        .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Units=("Quantity", "sum"))
        .sort_values("Sales", ascending=False)
    )
    segment_perf["Profit Margin (%)"] = np.where(
        segment_perf["Sales"] != 0,
        segment_perf["Profit"] / segment_perf["Sales"] * 100,
        np.nan
    )
    display(segment_perf)
else:
    print("Segment column not available in this dataset.")


## 9. Discount and profitability relationship

In [ ]:
if "Discount" in analysis_df.columns:
    discount_perf = (
        analysis_df.groupby("Discount", as_index=False)
        .agg(Orders_or_rows=("Sales", "size"), Sales=("Sales", "sum"), Profit=("Profit", "sum"))
        .sort_values("Discount")
    )
    discount_perf["Profit Margin (%)"] = np.where(
        discount_perf["Sales"] != 0,
        discount_perf["Profit"] / discount_perf["Sales"] * 100,
        np.nan
    )
    display(discount_perf)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(analysis_df["Discount"], analysis_df["Profit"], alpha=0.35)
    ax.axhline(0, linewidth=1)
    ax.set_title("Order-line Discount vs Profit")
    ax.set_xlabel("Discount")
    ax.set_ylabel("Profit")
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()
else:
    print("Discount column not available.")


## 10. Top products and customers

In [ ]:
if "Product Name" in analysis_df.columns:
    top_products = (
        analysis_df.groupby("Product Name", as_index=False)
        .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"))
        .sort_values("Sales", ascending=False)
        .head(10)
    )
    display(top_products)

if "Customer Name" in analysis_df.columns:
    customer_perf = (
        analysis_df.groupby("Customer Name", as_index=False)
        .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"))
        .sort_values("Sales", ascending=False)
        .head(10)
    )
    display(customer_perf)


## 11. Automated observations for review
These observations are calculated from the loaded data. They are descriptive, not proof of causation.

In [ ]:
observations = []

best_sales_category = category_perf.sort_values("Sales", ascending=False).iloc[0]
best_profit_category = category_perf.sort_values("Profit", ascending=False).iloc[0]
worst_subcategory = subcategory_perf.iloc[0]
best_region_sales = region_perf.sort_values("Sales", ascending=False).iloc[0]
worst_region_profit = region_perf.sort_values("Profit").iloc[0]

observations.append(
    f"Highest-sales category: {best_sales_category['Category']} "
    f"(sales = {best_sales_category['Sales']:,.2f})."
)
observations.append(
    f"Highest-profit category: {best_profit_category['Category']} "
    f"(profit = {best_profit_category['Profit']:,.2f})."
)
observations.append(
    f"Lowest-profit sub-category in this dataset: {worst_subcategory['Sub-Category']} "
    f"(profit = {worst_subcategory['Profit']:,.2f})."
)
observations.append(
    f"Highest-sales region: {best_region_sales['Region']} "
    f"(sales = {best_region_sales['Sales']:,.2f})."
)
observations.append(
    f"Lowest-profit region: {worst_region_profit['Region']} "
    f"(profit = {worst_region_profit['Profit']:,.2f})."
)
observations.append(
    f"Overall profit margin: {profit_margin:.2f}%." if pd.notna(profit_margin)
    else "Overall profit margin could not be calculated."
)

for i, item in enumerate(observations, start=1):
    print(f"{i}. {item}")

print("\nSuggested business questions for follow-up:")
print("- Are low-profit sub-categories affected by high discounts, shipping costs, or product costs?")
print("- Which months show repeatable seasonal patterns, and do they also produce profit?")
print("- Should marketing investment prioritize high-profit segments rather than sales alone?")


## 12. Export clean data and summary tables
The output files are saved in the current working directory so they can be included in a project submission or reviewed later.

In [ ]:
analysis_df.to_csv("cleaned_superstore_data.csv", index=False)
kpis.to_csv("kpi_summary.csv", index=False)
monthly.to_csv("monthly_sales_profit.csv", index=False)
category_perf.to_csv("category_performance.csv", index=False)
region_perf.to_csv("region_performance.csv", index=False)

print("Exported: cleaned_superstore_data.csv, kpi_summary.csv, monthly_sales_profit.csv,")
print("          category_performance.csv, region_performance.csv")


## 13. Conclusion and limitations
Use the computed tables and charts to write a short evidence-based conclusion after running the notebook.

**Recommended decision approach**
1. Review profit as well as sales before deciding which products or regions to prioritize.
2. Investigate loss-making sub-categories and discount levels before changing prices or promotions.
3. Compare monthly sales and profit to avoid treating revenue growth as the only success measure.
4. Validate findings with operational context such as shipping costs, inventory, and campaign calendars.

**Limitations**
- This is a sample retail dataset, not necessarily a company's current or complete transaction history.
- Relationships between discounts and profit are descriptive; they do not establish causation.
- Results depend on the source dataset and the cleaning rules above.
- AI can help explain patterns and brainstorm follow-up questions, but business recommendations should be checked against the underlying numbers and domain context.
